In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/multigraph_manual")
metadata_dir = save_dir / "metadata"

gauges = gpd.read_parquet(metadata_dir / "gauges.parquet").set_index('site_id')

subs_df = pd.read_parquet(metadata_dir / 'subbasins.parquet', columns=['site_id', 'is_gauge', 'uparea_km2', 'outlet_id'])
subs_df.index = subs_df.index.astype(str)

basin_subbasin_dict = subs_df.groupby('outlet_id').apply(lambda g: g.index.tolist(), include_groups=False).to_dict()

In [2]:
from data import ZarrBasinStore
store_path = Path("/scratch4/workspace/tlanghorst_umass_edu-swot-ml-zarr/zbs_batched")
store = ZarrBasinStore(store_path)

In [3]:
import global_gauges as gg
facade = gg.GaugeDataFacade()

In [4]:
facade.get_daily_values('EAUF-E4905710')

FileNotFoundError: No such file or directory (os error 2): /nas/cee-ice/data/gg_parquet/eauf/discharge/site_id=EAUF-E4905710/data.parquet

This error occurred with the following context stack:
	[1] 'parquet scan'
	[2] 'sink'


In [ ]:
BATCH_SIZE = 50 
for basin_id, basin_group in tqdm(subs_df.groupby('outlet_id')):
    basin_subs = basin_subbasin_dict[basin_id]
    n_subs = len(basin_subs)
    
    for i in range(0, n_subs, BATCH_SIZE):
        batch_subs = basin_subs[i : i + BATCH_SIZE]
        
        batch_df_list = []
        for subbasin in batch_subs:
            sub_row = basin_group.loc[subbasin]
            if not sub_row['is_gauge']:
                continue

            try:
                gauge_df = facade.get_daily_values(subbasin, start_date='1980-01-01', end_date='2024-12-31')
            except FileNotFoundError:
                continue

                
            if gauge_df.empty:
                continue

            gauge_df = gauge_df.loc[~gauge_df.index.duplicated(keep='first'), :]
            gauge_df = gauge_df[['discharge']].droplevel(axis=0, level=0)
            gauge_df['unit_discharge'] = gauge_df / sub_row['uparea_km2']

            if gauge_df is not None and not gauge_df.empty:
                gauge_df['subbasin'] = subbasin
                batch_df_list.append(gauge_df)


        # If the whole batch is empty, we technically don't need to write anything 
        # since the zarr was initialized with NaNs/empty).
        if batch_df_list:
            batch_df = pd.concat(batch_df_list)
            store.write_batch_data(basin_id, batch_df, basin_subs, batch_subs)
        

100%|██████████| 1472/1472 [17:16<00:00,  1.42it/s]  


In [ ]:
debug

In [12]:
gauge_df

,discharge,unit_discharge,subbasin
date,,,
2013-08-23,6.4800,0.001875,BRANA-13540000
2013-08-24,8.3800,0.002425,BRANA-13540000
2013-08-25,12.3500,0.003574,BRANA-13540000
2013-08-26,9.0100,0.002608,BRANA-13540000
2013-08-27,8.5400,0.002472,BRANA-13540000
...,...,...,...
2024-12-27,53.0155,0.015343,BRANA-13540000
2024-12-28,53.7966,0.015569,BRANA-13540000
2024-12-29,43.8568,0.012693,BRANA-13540000


,discharge,unit_discharge,subbasin
date,,,
2013-08-23,6.4800,0.001875,BRANA-13540000
2013-08-24,8.3800,0.002425,BRANA-13540000
2013-08-25,12.3500,0.003574,BRANA-13540000
2013-08-26,9.0100,0.002608,BRANA-13540000
2013-08-27,8.5400,0.002472,BRANA-13540000
...,...,...,...
2024-12-27,53.0155,0.015343,BRANA-13540000
2024-12-28,53.7966,0.015569,BRANA-13540000
2024-12-29,43.8568,0.012693,BRANA-13540000
